# **Gradient Boosting Algorithms**

## **XGBoost**

XGBoost (eXtreme Gradient Boosting) is a powerful machine learning algorithm that is based on gradient boosting. Developed by Tianqi Chen, it has become one of the most popular algorithms for structured or tabular data due to its high performance and efficiency. XGBoost works by building an ensemble of decision trees in a sequential manner, where each tree attempts to correct the errors of the previous one. This iterative approach helps in minimizing the loss function and improving the model's predictions.

### **XGBoost Hyperparameters**

1. **n_estimators**: Number of boosting rounds.
   - Example values: `100`, `200`, `500`, `1000`

2. **learning_rate (eta)**: Step size shrinkage used in updates to prevent overfitting.
   - Example values: `0.01`, `0.1`, `0.2`, `0.3`

3. **max_depth**: Maximum depth of a tree.
   - Example values: `3`, `6`, `9`, `12`

4. **min_child_weight**: Minimum sum of instance weight (hessian) needed in a child.
   - Example values: `1`, `5`, `10`, `15`

5. **gamma**: Minimum loss reduction required to make a further partition on a leaf node of the tree.
   - Example values: `0`, `0.1`, `1`, `5`

6. **subsample**: Subsample ratio of the training instances.
   - Example values: `0.5`, `0.7`, `0.8`, `1.0`

7. **colsample_bytree**: Subsample ratio of columns when constructing each tree.
   - Example values: `0.3`, `0.5`, `0.7`, `1.0`

8. **colsample_bylevel**: Subsample ratio of columns for each level.
   - Example values: `0.3`, `0.5`, `0.7`, `1.0`

9. **colsample_bynode**: Subsample ratio of columns for each node (split).
   - Example values: `0.3`, `0.5`, `0.7`, `1.0`

10. **scale_pos_weight**: Control the balance of positive and negative weights, useful for unbalanced classes.
    - Example values: `1`, `5`, `10`, `20` (values are typically the ratio of negative to positive samples in the dataset)

11. **reg_alpha**: L1 regularization term on weights (Lasso).
    - Example values: `0`, `0.1`, `1`, `10`

12. **reg_lambda**: L2 regularization term on weights (Ridge).
    - Example values: `0`, `1`, `10`, `100`



## **LightGBM**
LightGBM (Light Gradient Boosting Machine) is an efficient and highly scalable implementation of gradient boosting developed by Microsoft. It is designed to be distributed and efficient.
LightGBM builds decision trees sequentially, similar to other gradient boosting frameworks.

### **LightGBM Hyperparameters**

1. **n_estimators**: Number of boosting rounds.
   - Example values: `100`, `200`, `500`, `1000`

2. **learning_rate**: Step size shrinkage used in updates to prevent overfitting.
   - Example values: `0.01`, `0.05`, `0.1`, `0.2`

3. **num_leaves**: Maximum tree leaves for base learners.
   - Example values: `31`, `50`, `100`, `255`

4. **max_depth**: Maximum depth of the tree.
   - Example values: `-1` (no limit), `6`, `10`, `15`

5. **min_data_in_leaf**: Minimum number of data needed in a leaf.
   - Example values: `20`, `50`, `100`, `200`

6. **bagging_fraction**: Fraction of data to be used for each iteration (randomly selected).
   - Example values: `0.5`, `0.7`, `0.8`, `1.0`

7. **feature_fraction**: Fraction of features to be used for each iteration (randomly selected).
   - Example values: `0.6`, `0.7`, `0.8`, `1.0`

8. **lambda_l1**: L1 regularization term on weights.
   - Example values: `0`, `0.1`, `1`, `10`

9. **lambda_l2**: L2 regularization term on weights.
   - Example values: `0`, `0.1`, `1`, `10`

10. **min_gain_to_split**: Minimum gain to make a split.
    - Example values: `0`, `0.1`, `1`, `10`

11. **scale_pos_weight**: Control the balance of positive and negative weights, useful for unbalanced classes.
    - Example values: `1`, `5`, `10`, `20` (values are typically the ratio of negative to positive samples in the dataset)



## **HyperParameter Tuning**

### **GridSearchCV**

GridSearchCV is a hyperparameter optimization technique provided by scikit-learn, a popular machine learning library in Python. It is used to find the best combination of hyperparameters for a given model by exhaustively searching through a specified grid of possible parameter values. This process helps in enhancing the model's performance by selecting the optimal set of hyperparameters.

### **RandomizedSearchCV**

RandomizedSearchCV is an alternative to GridSearchCV for hyperparameter tuning provided by scikit-learn. Unlike GridSearchCV, which exhaustively searches through all possible combinations of hyperparameters, RandomizedSearchCV samples a fixed number of parameter settings from specified distributions. This approach can significantly reduce the computation time and still find a good combination of hyperparameters.

In [23]:
!pip install xgboost

In [24]:
!pip install lightgbm

In [25]:
pip install --upgrade lightgbm dask

Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score
from sklearn.svm import SVC
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import uniform, randint
import scipy.stats as stats

## **XGBoost**

### **OOP Approach**

In [27]:

class ChurnPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    #  Load data
    def load_data(self):
        self.data = pd.read_excel(self.file_path)   # FIXED
        print("Data loaded!")

    #  Preprocess
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]

        self.X = self.data[selected_features]
        self.y = self.data['Exited']   # FIXED (1D)

        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)
        print("Preprocessing done!")

    # Split
    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=0.8, random_state=42
        )
        print("Data split!")

    #  Train XGBoost
    def train_model(self):
        self.model = xgb.XGBClassifier(
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        )
        self.model.fit(self.train_X, self.train_y)
        print("Model trained!")

    #  Evaluate
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)

        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print("Model saved!")

    def load_model(self, model_path):
        self.model = joblib.load(model_path)
        print("Model loaded!")


# =========================
#  Usage
# =========================

churn = ChurnPrediction('Churn.xlsx')  # file in notebook
churn.load_data()
churn.preprocess_data()
churn.split_data()
churn.train_model()
accuracy, auc = churn.evaluate_model()

churn.save_model('churn_model.pkl')

Data loaded!
Preprocessing done!
Data split!
Model trained!
Model accuracy: 0.8695
Model AUC score: 0.8502
Model saved!


### **Procedural Approach for Xgboost with Automated Hyperparameter Tuning via GridSearchCV**

In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import xgboost as xgb


# ======================
# Load data
# ======================
def load_data(file_path):
    return pd.read_excel(file_path)   # FIXED


# ======================
# Preprocess
# ======================
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]
    y = data['Exited']   # FIXED

    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


# ======================
# Split
# ======================
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


# ======================
# Train XGBoost with tuning
# ======================
def train_model_xgb(train_X, train_y):

    model = xgb.XGBClassifier(
        random_state=0,
        use_label_encoder=False,
        eval_metric='logloss'
    )

    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5, 7],
    }

    grid_search = GridSearchCV(
        model,
        param_grid,
        scoring='roc_auc',
        cv=3,
        n_jobs=-1   # faster 
    )

    grid_search.fit(train_X, train_y)

    print("Best Params:", grid_search.best_params_)
    return grid_search.best_estimator_


# ======================
# Evaluate
# ======================
def evaluate_model(model, val_X, val_y):
    val_prediction = model.predict(val_X)

    accuracy = accuracy_score(val_y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    y_pred_proba = model.predict_proba(val_X)[:, 1]
    auc = roc_auc_score(val_y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# ======================
# Save
# ======================
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


# ======================
#  Usage
# ======================
file_path = 'Churn.xlsx'

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

model_xgb = train_model_xgb(train_X, train_y)

accuracy_xgb, auc_xgb = evaluate_model(model_xgb, val_X, val_y)

save_model(model_xgb, 'best_xgb_model.pkl')

Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Model accuracy: 0.8625
Model AUC score: 0.8738
Model saved!


### **Procedural Approach for Xgboost with Automated Hyperparameter Tuning via RandomizedSearchCV**

In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import xgboost as xgb
from scipy.stats import uniform, randint


# ======================
# Load data
# ======================
def load_data(file_path):
    return pd.read_excel(file_path)   # FIXED


# ======================
# Preprocess
# ======================
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]
    y = data['Exited']   # FIXED

    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


# ======================
# Split
# ======================
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


# ======================
# Train XGBoost (Random Search)
# ======================
def train_model_xgb(train_X, train_y):

    model = xgb.XGBClassifier(
        random_state=0,
        use_label_encoder=False,
        eval_metric='logloss'
    )

    param_dist = {
        'n_estimators': randint(100, 300),
        'learning_rate': uniform(0.01, 0.1),
        'max_depth': randint(3, 7),
        'min_child_weight': randint(1, 5),
        'gamma': uniform(0, 0.3),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3),
        'reg_alpha': uniform(0, 0.1),
        'reg_lambda': uniform(1, 1)
    }

    random_search = RandomizedSearchCV(
        model,
        param_distributions=param_dist,
        scoring='roc_auc',
        cv=3,
        n_iter=50,
        random_state=0,
        n_jobs=-1   # faster
    )

    random_search.fit(train_X, train_y)

    print("Best Params:", random_search.best_params_)
    return random_search.best_estimator_


# ======================
# Evaluate
# ======================
def evaluate_model(model, val_X, val_y):
    val_prediction = model.predict(val_X)

    accuracy = accuracy_score(val_y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    y_pred_proba = model.predict_proba(val_X)[:, 1]
    auc = roc_auc_score(val_y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# ======================
# Save
# ======================
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


# ======================
#  Usage
# ======================
file_path = 'Churn.xlsx'

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

model_xgb = train_model_xgb(train_X, train_y)

accuracy_xgb, auc_xgb = evaluate_model(model_xgb, val_X, val_y)

save_model(model_xgb, 'best_xgb_model.pkl')

Best Params: {'colsample_bytree': 0.7117563376762962, 'gamma': 0.08484208877292287, 'learning_rate': 0.022019656121316893, 'max_depth': 6, 'min_child_weight': 4, 'n_estimators': 230, 'reg_alpha': 0.06886611828057704, 'reg_lambda': 1.8804758892525955, 'subsample': 0.9754706399086434}
Model accuracy: 0.8665
Model AUC score: 0.8788
Model saved!


## **LightGBM Implementation**

### **OOP Approach**

In [30]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib


class ChurnPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    #  Load data
    def load_data(self):
        self.data = pd.read_excel(self.file_path)   # FIXED
        print("Data loaded!")

    #  Preprocess
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]

        self.X = self.data[selected_features]
        self.y = self.data['Exited']   # FIXED

        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)
        print("Preprocessing done!")

    #  Split
    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=0.8, random_state=0
        )
        print("Data split!")

    #  Train LightGBM
    def train_model(self):
        self.model = lgb.LGBMClassifier(random_state=0)
        self.model.fit(self.train_X, self.train_y)
        print("Model trained!")

    #  Evaluate
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)

        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print("Model saved!")

    def load_model(self, model_path):
        self.model = joblib.load(model_path)
        print("Model loaded!")


# =========================
# Usage
# =========================

churn = ChurnPrediction('Churn.xlsx')
churn.load_data()
churn.preprocess_data()
churn.split_data()
churn.train_model()
accuracy, auc = churn.evaluate_model()

churn.save_model('churn_model.pkl')

Data loaded!
Preprocessing done!
Data split!
[LightGBM] [Info] Number of positive: 1632, number of negative: 6368
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000922 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 856
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.204000 -> initscore=-1.361479
[LightGBM] [Info] Start training from score -1.361479
Model trained!
Model accuracy: 0.8635
Model AUC score: 0.8685
Model saved!


### **Procedural Approach for LightGBM with Automated Hyperparameter Tuning via GridSearchCV**

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import lightgbm as lgb


# ======================
# Load data
# ======================
def load_data(file_path):
    return pd.read_excel(file_path)   # FIXED


# ======================
# Preprocess
# ======================
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]
    y = data['Exited']   # FIXED

    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


# ======================
# Split
# ======================
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


# ======================
# Train LightGBM
# ======================
def train_model_lgb(train_X, train_y):

    model = lgb.LGBMClassifier(random_state=0)

    param_grid = {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.1],
        'num_leaves': [31, 50]
    }

    grid_search = GridSearchCV(
        model,
        param_grid,
        scoring='roc_auc',
        cv=3,
        n_jobs=-1   #  faster
    )

    grid_search.fit(train_X, train_y)

    print("Best Params:", grid_search.best_params_)
    return grid_search.best_estimator_


# ======================
# Evaluate
# ======================
def evaluate_model(model, val_X, val_y):
    val_prediction = model.predict(val_X)

    accuracy = accuracy_score(val_y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    y_pred_proba = model.predict_proba(val_X)[:, 1]
    auc = roc_auc_score(val_y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# ======================
# Save
# ======================
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


# ======================
#  Usage
# ======================
file_path = 'Churn.xlsx'

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

model_lgb = train_model_lgb(train_X, train_y)

accuracy_lgb, auc_lgb = evaluate_model(model_lgb, val_X, val_y)

save_model(model_lgb, 'best_lgb_model.pkl')

[LightGBM] [Info] Number of positive: 1632, number of negative: 6368
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 856
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.204000 -> initscore=-1.361479
[LightGBM] [Info] Start training from score -1.361479
Best Params: {'learning_rate': 0.01, 'n_estimators': 500, 'num_leaves': 31}
Model accuracy: 0.8635
Model AUC score: 0.8748
Model saved!


### **Procedural Approach for LightGBM with Automated Hyperparameter Tuning via RandomizedSearchCV**

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import lightgbm as lgb
from scipy.stats import randint, uniform


# ======================
# Load data
# ======================
def load_data(file_path):
    return pd.read_excel(file_path)   # FIXED


# ======================
# Preprocess
# ======================
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]
    y = data['Exited']   # FIXED

    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


# ======================
# Split
# ======================
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


# ======================
# Train LightGBM (TRUE Random Search)
# ======================
def train_model_lgb(train_X, train_y):

    model = lgb.LGBMClassifier(random_state=0)

    param_dist = {
        'n_estimators': randint(100, 500),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 80)
    }

    random_search = RandomizedSearchCV(
        model,
        param_distributions=param_dist,
        scoring='roc_auc',
        cv=3,
        n_iter=30,        # slightly reduced (faster)
        random_state=0,
        n_jobs=-1         # speed
    )

    random_search.fit(train_X, train_y)

    print("Best Params:", random_search.best_params_)
    return random_search.best_estimator_


# ======================
# Evaluate
# ======================
def evaluate_model(model, val_X, val_y):
    val_prediction = model.predict(val_X)

    accuracy = accuracy_score(val_y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    y_pred_proba = model.predict_proba(val_X)[:, 1]
    auc = roc_auc_score(val_y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# ======================
# Save
# ======================
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


# ======================
#  Usage
# ======================
file_path = 'Churn.xlsx'

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

model_lgb = train_model_lgb(train_X, train_y)

accuracy_lgb, auc_lgb = evaluate_model(model_lgb, val_X, val_y)

save_model(model_lgb, 'best_lgb_model.pkl')

[LightGBM] [Info] Number of positive: 1632, number of negative: 6368
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000219 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 856
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.204000 -> initscore=-1.361479
[LightGBM] [Info] Start training from score -1.361479
Best Params: {'learning_rate': 0.013842542647273474, 'n_estimators': 303, 'num_leaves': 24}
Model accuracy: 0.8660
Model AUC score: 0.8752
Model saved!


## **Integration of both algorithms**

### **OOP Approach using XGBoost and LightGBM with Automated Hyperparameter Tuning via GridSearchCV**

In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import xgboost as xgb
import lightgbm as lgb

class ChurnPrediction:
    def __init__(self, file_path, model_type='xgboost'):
        """
        file_path: str, path to CSV or Excel file
        model_type: str, 'xgboost' or 'lightgbm'
        """
        self.file_path = file_path
        self.model_type = model_type
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    # ======================
    # Load data (CSV or Excel)
    # ======================
    def load_data(self):
        if self.file_path.endswith('.csv'):
            self.data = pd.read_csv(self.file_path)
        elif self.file_path.endswith(('.xls', '.xlsx')):
            self.data = pd.read_excel(self.file_path)
        else:
            raise ValueError("File must be .csv or .xlsx")
        print("Data loaded!")

    # ======================
    # Preprocess
    # ======================
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]

        self.X = self.data[selected_features]
        self.y = self.data['Exited']  # target column

        # One-hot encoding (drop first to avoid multicollinearity)
        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)

        # Feature scaling
        scaler = StandardScaler()
        self.X = scaler.fit_transform(self.X)

        print("Preprocessing done!")

    # ======================
    # Split data
    # ======================
    def split_data(self, train_size=0.8):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=train_size, random_state=0
        )
        print("Data split into train and validation sets!")

    # ======================
    # Train model
    # ======================
    def train_model(self):
        if self.model_type == 'xgboost':
            base_model = xgb.XGBClassifier(
                random_state=0,
                use_label_encoder=False,
                eval_metric='logloss'
            )
            param_grid = {
                'n_estimators': [100, 200],
                'learning_rate': [0.01, 0.1],
                'max_depth': [3, 5, 7]
            }

        elif self.model_type == 'lightgbm':
            base_model = lgb.LGBMClassifier(random_state=0)
            param_grid = {
                'n_estimators': [100, 200],
                'learning_rate': [0.01, 0.1],
                'num_leaves': [31, 50]
            }
        else:
            raise ValueError("model_type must be 'xgboost' or 'lightgbm'")

        # GridSearchCV for hyperparameter tuning
        grid_search = GridSearchCV(
            base_model,
            param_grid,
            scoring='roc_auc',
            cv=3,
            n_jobs=-1
        )
        grid_search.fit(self.train_X, self.train_y)
        self.model = grid_search.best_estimator_
        print("Training done!")
        print("Best Params:", grid_search.best_params_)

    # ======================
    # Evaluate model
    # ======================
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)
        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    # ======================
    # Save / Load model
    # ======================
    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print(f"Model saved to {model_path}!")

    def load_model(self, model_path):
        self.model = joblib.load(model_path)
        print(f"Model loaded from {model_path}!")

# ======================
# Example Usage
# ======================

file_path_ = 'Churn.xlsx'  # your Excel 

# ---- XGBoost ----
churn_xgb = ChurnPrediction(file_path_, model_type='xgboost')
churn_xgb.load_data()
churn_xgb.preprocess_data()
churn_xgb.split_data()
churn_xgb.train_model()
accuracy_xgb, auc_xgb = churn_xgb.evaluate_model()
churn_xgb.save_model('churn_xgb_model.pkl')

# ---- LightGBM ----
churn_lgb = ChurnPrediction(file_path_, model_type='lightgbm')
churn_lgb.load_data()
churn_lgb.preprocess_data()
churn_lgb.split_data()
churn_lgb.train_model()
accuracy_lgb, auc_lgb = churn_lgb.evaluate_model()
churn_lgb.save_model('churn_lgb_model.pkl')

Data loaded!
Preprocessing done!
Data split into train and validation sets!
Training done!
Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Model accuracy: 0.8625
Model AUC score: 0.8738
Model saved to churn_xgb_model.pkl!
Data loaded!
Preprocessing done!
Data split into train and validation sets!
[LightGBM] [Info] Number of positive: 1632, number of negative: 6368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000682 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 862
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.204000 -> initscore=-1.361479
[LightGBM] [Info] Start training from score -1.361479
Training done!
Best Params: {'learning_rate': 0.1, 'n_estimators': 100, 'num_leaves': 31}
Model accuracy: 0.8595
Model AUC score: 0.8690
Model saved to churn_lgb_model.pkl!


### **Procedural Approach using XGBoost and LightGBM with Automated Hyperparameter Tuning via RandomizedSearchCV**

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import xgboost as xgb
import lightgbm as lgb
from scipy import stats  # needed for random distributions

class ChurnPrediction:
    def __init__(self, file_path, model_type='xgboost'):
        self.file_path = file_path
        self.model_type = model_type
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    # ----------------------
    # Load data (CSV or Excel)
    # ----------------------
    def load_data(self):
        if self.file_path.endswith('.csv'):
            self.data = pd.read_csv(self.file_path)
        elif self.file_path.endswith(('.xls', '.xlsx')):
            self.data = pd.read_excel(self.file_path)
        else:
            raise ValueError("File must be .csv or .xlsx")
        print("Data loaded!")

    # ----------------------
    # Preprocess
    # ----------------------
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]
        self.X = self.data[selected_features]
        self.y = self.data['Exited']

        # One-hot encoding, drop_first to avoid dummy trap
        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)

        # Feature scaling
        scaler = StandardScaler()
        self.X = scaler.fit_transform(self.X)
        print("Preprocessing done!")

    # ----------------------
    # Split data
    # ----------------------
    def split_data(self, train_size=0.8):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=train_size, random_state=0
        )
        print("Data split into train and validation sets!")

    # ----------------------
    # Train model
    # ----------------------
    def train_model(self):
        if self.model_type == 'xgboost':
            self.model = xgb.XGBClassifier(random_state=0, use_label_encoder=False, eval_metric='logloss')
            param_dist = {
                'n_estimators': stats.randint(100, 300),
                'learning_rate': stats.uniform(0.01, 0.1),
                'max_depth': stats.randint(3, 7)
            }

        elif self.model_type == 'lightgbm':
            self.model = lgb.LGBMClassifier(random_state=0)
            param_dist = {
                'n_estimators': stats.randint(100, 300),
                'learning_rate': stats.uniform(0.01, 0.1),
                'num_leaves': stats.randint(31, 70)
            }
        else:
            raise ValueError("model_type must be 'xgboost' or 'lightgbm'")

        random_search = RandomizedSearchCV(
            self.model,
            param_distributions=param_dist,
            n_iter=50,
            scoring='roc_auc',
            cv=3,
            n_jobs=-1,
            random_state=0
        )
        random_search.fit(self.train_X, self.train_y.values.ravel())

        self.model = random_search.best_estimator_
        print("Training done! Best params:", random_search.best_params_)

    # ----------------------
    # Evaluate
    # ----------------------
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)
        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    # ----------------------
    # Save / Load
    # ----------------------
    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print(f"Model saved to {model_path}!")

    def load_model(self, model_path):
        self.model = joblib.load(model_path)
        print(f"Model loaded from {model_path}!")

# ======================
# Example Usage
# ======================
file_path_ = 'Churn.xlsx'  # replace with your file path

# XGBoost
churn_xgb = ChurnPrediction(file_path_, model_type='xgboost')
churn_xgb.load_data()
churn_xgb.preprocess_data()
churn_xgb.split_data()
churn_xgb.train_model()
accuracy_xgb, auc_xgb = churn_xgb.evaluate_model()
churn_xgb.save_model('churn_xgb_model.pkl')

# LightGBM
churn_lgb = ChurnPrediction(file_path_, model_type='lightgbm')
churn_lgb.load_data()
churn_lgb.preprocess_data()
churn_lgb.split_data()
churn_lgb.train_model()
accuracy_lgb, auc_lgb = churn_lgb.evaluate_model()
churn_lgb.save_model('churn_lgb_model.pkl')

Data loaded!
Preprocessing done!
Data split into train and validation sets!
Training done! Best params: {'learning_rate': 0.04380076148388918, 'max_depth': 4, 'n_estimators': 185}
Model accuracy: 0.8695
Model AUC score: 0.8778
Model saved to churn_xgb_model.pkl!
Data loaded!
Preprocessing done!
Data split into train and validation sets!
[LightGBM] [Info] Number of positive: 1632, number of negative: 6368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000702 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 862
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.204000 -> initscore=-1.361479
[LightGBM] [Info] Start training from score -1.361479
Training done! Best params: {'learning_rate': 0.06089689606670145, 'n_estimators': 114, 'num_leaves': 32}
Model accuracy: 0.8635
Model AUC score: 0.8730
Model saved to chu